# CertGen CIFAR-10 Feature Extraction T4x2 1k

This notebook creates feature-cache artifacts only. It does not run certificates, metric reproduction claims, pilot undecided fraction, or paper evidence generation.

`claim_allowed=false`  
`NO_FAKE_RESULTS`  
`NO_REAL_EVIDENCE until gates pass`  
`not paper evidence`

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
print('python', sys.version)
try:
    import torch
    print('cuda_available', torch.cuda.is_available())
    print('gpu_count', torch.cuda.device_count())
    print('gpu_names', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception as exc:
    print('torch_check_failed', exc)
print('disk_usage', shutil.disk_usage('/kaggle/working'))

In [ ]:
!pip -q install torch torchvision transformers timm pillow numpy scipy PyYAML

In [ ]:
INPUT_ZIP = Path('/kaggle/input/certgen-features/certgen_cifar10_feature_extraction_1k_input.zip')
WORK = Path('/kaggle/working/certgen_feature_input')
WORK.mkdir(parents=True, exist_ok=True)
if not INPUT_ZIP.exists():
    raise FileNotFoundError(f'Missing Kaggle input ZIP: {INPUT_ZIP}')
!unzip -q -o {INPUT_ZIP} -d {WORK}
config = json.loads((WORK / 'config/feature_extraction_config.json').read_text())
assert config['claim_allowed'] is False
for required in ['manifests/cifar10_r1_feature_extraction_samples.jsonl', 'inputs/cifar10_r1_ledger.csv']:
    assert (WORK / required).exists(), required
config

In [ ]:
FEATURE_ROOT = Path('/kaggle/working/features/cifar10_r1')
for p in [FEATURE_ROOT / 'inception', FEATURE_ROOT / 'clip', FEATURE_ROOT / 'merged', FEATURE_ROOT / 'split', FEATURE_ROOT / 'logs']:
    p.mkdir(parents=True, exist_ok=True)
SAMPLE_MANIFEST = WORK / 'manifests/cifar10_r1_feature_extraction_samples.jsonl'
PROVENANCE = WORK / 'inputs/cifar10_r1_ledger.csv'
LOCKS = list((WORK / 'configs').glob('*.json'))
assert LOCKS, 'preprocessing lock missing'
LOCK = LOCKS[0]
print('sample_manifest', SAMPLE_MANIFEST)
print('preprocessing_lock', LOCK)

In [ ]:
def run_sharded(extractor, out_subdir, batch_size):
    commands = [
        ['bash', '-lc', f'CUDA_VISIBLE_DEVICES=0 python -m certgen.features.extract --input-manifest {SAMPLE_MANIFEST} --provenance-ledger {PROVENANCE} --preprocessing-lock {LOCK} --extractor {extractor} --out-dir {FEATURE_ROOT/out_subdir} --device cuda --batch-size {batch_size} --shard-id 0 --num-shards 2 --resume --execute > {FEATURE_ROOT}/logs/{extractor}_gpu0.log 2>&1'],
        ['bash', '-lc', f'CUDA_VISIBLE_DEVICES=1 python -m certgen.features.extract --input-manifest {SAMPLE_MANIFEST} --provenance-ledger {PROVENANCE} --preprocessing-lock {LOCK} --extractor {extractor} --out-dir {FEATURE_ROOT/out_subdir} --device cuda --batch-size {batch_size} --shard-id 1 --num-shards 2 --resume --execute > {FEATURE_ROOT}/logs/{extractor}_gpu1.log 2>&1'],
    ]
    start = time.time()
    procs = [subprocess.Popen(cmd) for cmd in commands]
    codes = [proc.wait() for proc in procs]
    wall = time.time() - start
    if any(code != 0 for code in codes):
        status = {'status_code': 'BLOCKED_FEATURE_EXTRACTION_FAILED', 'extractor': extractor, 'codes': codes, 'wall_time_seconds': wall, 'claim_allowed': False}
        Path('/kaggle/working/feature_extraction_blocked_status.json').write_text(json.dumps(status, indent=2))
        raise RuntimeError(status)
    return {'extractor': extractor, 'wall_time_seconds': wall, 'status': 'extracted', 'claim_allowed': False}

run_log = [run_sharded('inception_v3_pool3', 'inception', 64), run_sharded('clip_vit', 'clip', 64)]
Path('/kaggle/working/feature_run_log.json').write_text(json.dumps({'runs': run_log, 'evidence_status': 'run_log_only', 'claim_allowed': False}, indent=2))
run_log

In [ ]:
!python -m certgen.features.merge_shards --shard-dir /kaggle/working/features/cifar10_r1/inception/shard-000-of-002 --shard-dir /kaggle/working/features/cifar10_r1/inception/shard-001-of-002 --extractor inception_v3_pool3 --out-npz /kaggle/working/features/cifar10_r1/cifar10_r1_inception.npz --out-sidecar /kaggle/working/features/cifar10_r1/cifar10_r1_inception.sidecar.json --force
!python -m certgen.features.merge_shards --shard-dir /kaggle/working/features/cifar10_r1/clip/shard-000-of-002 --shard-dir /kaggle/working/features/cifar10_r1/clip/shard-001-of-002 --extractor clip_vit --out-npz /kaggle/working/features/cifar10_r1/cifar10_r1_clip.npz --out-sidecar /kaggle/working/features/cifar10_r1/cifar10_r1_clip.sidecar.json --force
!python -m certgen.features.split_by_role --features-npz /kaggle/working/features/cifar10_r1/cifar10_r1_inception.npz --sidecar /kaggle/working/features/cifar10_r1/cifar10_r1_inception.sidecar.json --sample-manifest {SAMPLE_MANIFEST} --extractor-label inception --out-dir /kaggle/working/features/cifar10_r1/split --summary-out /kaggle/working/features/cifar10_r1/split/inception_split_summary.json --force
!python -m certgen.features.split_by_role --features-npz /kaggle/working/features/cifar10_r1/cifar10_r1_clip.npz --sidecar /kaggle/working/features/cifar10_r1/cifar10_r1_clip.sidecar.json --sample-manifest {SAMPLE_MANIFEST} --extractor-label clip --out-dir /kaggle/working/features/cifar10_r1/split --summary-out /kaggle/working/features/cifar10_r1/split/clip_split_summary.json --force

In [ ]:
for required in ['reference_inception.npz', 'google_ddpm_inception.npz', 'frank_ddpm_ema_inception.npz', 'frank_cfm_inception.npz', 'reference_clip.npz', 'google_ddpm_clip.npz', 'frank_ddpm_ema_clip.npz', 'frank_cfm_clip.npz']:
    assert (FEATURE_ROOT / 'split' / required).exists(), required
OUTPUT_ZIP = Path('/kaggle/working/certgen_cifar10_features_1k_outputs.zip')
!cd /kaggle/working && zip -qr {OUTPUT_ZIP} features/cifar10_r1 feature_run_log.json
print('Copy back this ZIP to local data/kaggle_outputs/:', OUTPUT_ZIP)
print('Then run: commands/v6_cpu_execution/07_validate_copied_back_feature_caches.sh')